# mailman - training the extraction model

Trains a token classifier that labels each word of an invoice with the field it belongs to,
then exports it for the mailman pipeline to serve locally. No API key, no per-call cost.

**Run this on Colab with a GPU.** Runtime -> Change runtime type -> T4 GPU. It is free, and
a run takes a few minutes.

## Why a tagger and not a generative model

The pipeline already has the document's text layer, and every field wanted is a span that
physically appears on the page. A tagger returns *where* it found each value, so a value it
returns came from the document. That removes a whole class of failure - a tagger cannot
invent an invoice number that was never printed. It can only mislabel one that was.

The cost is that it cannot infer anything not written down, and it has no idea what an
invoice *means*. Those are real limits and they belong in the README.

## What this produces

A directory of weights (~250 MB) that `mailman/trained.py` loads. The weights do **not** go
in git - too large - and will not fit a free hosting tier's memory. The heuristic extractor
is what deploys; this is the local and showcase path, and the interesting number is the gap
between the two.


In [ ]:
# What GPU did Colab give us, if any.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "No GPU. Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" seqeval torch --upgrade
print("installed")


## 1. Training data

The invoices are generated, which means **the labels come free**. The generator knows what
it printed and where, so every training example is labelled by construction rather than by
hand. This is the same reason the project's evaluation corpus is safe to build after the
pipeline rather than before it.

That is also this notebook's biggest weakness, and it should be said plainly: a model
trained only on invoices from one generator learns that generator. The layouts below vary on
purpose - different label wording, different orderings, different date and money formats -
but it is still one author's idea of what an invoice looks like. Real or public samples
mixed in are what would make the numbers mean something, and the place to do that is the
cell marked **Optional** further down.


In [ ]:
import random
from datetime import date, timedelta

FIELD_LABELS = [
    "INVOICE_NUMBER", "VENDOR_NAME", "BUYER_NAME", "ISSUE_DATE", "DUE_DATE",
    "CURRENCY", "SUBTOTAL", "TAX", "TOTAL",
    "LINE_DESCRIPTION", "LINE_QUANTITY", "LINE_UNIT_PRICE", "LINE_AMOUNT",
]
LABELS = ["O"] + [f"{p}-{f}" for f in FIELD_LABELS for p in ("B", "I")]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

VENDORS = ["Acme Corp Ltd", "Northgate Supplies", "Bluewater Logistics",
           "Harrow & Finch", "Tessellate Systems", "Meridian Print Works"]
BUYERS = ["Kestrel Retail Ltd", "Orchard Foods", "Vantage Media", "Pelham Group"]
GOODS = ["Widget assembly", "Freight charge", "Consulting hours", "Packaging",
         "Installation", "Annual licence", "Courier", "Site survey"]
CURRENCIES = [("GBP", "GBP "), ("USD", "$"), ("EUR", "EUR ")]

# The layout knobs. Each one is a way a real invoice differs from the one before it, and
# each one is a way the heuristic extractor can be wrong.
NUMBER_LABELS = ["Invoice Number:", "Invoice No.", "Invoice #", "INV NUMBER"]
DATE_LABELS = ["Invoice Date:", "Date of Issue:", "Date:", "Issued:"]
DUE_LABELS = ["Due Date:", "Payment Due:", "Due:"]
TOTAL_LABELS = ["Total Due", "Grand Total", "TOTAL", "Amount Due"]


def money(value, symbol):
    return f"{symbol}{value:,.2f}"


def a_date(d, style):
    if style == 0:
        return d.isoformat()
    if style == 1:
        return d.strftime("%d/%m/%Y")
    if style == 2:
        return d.strftime("%d %B %Y")
    return d.strftime("%b %d, %Y")


In [ ]:
def generate_invoice(rng):
    """Return (words, labels) for one invoice.

    Labels are attached as the text is written, not matched afterwards. Matching a value
    back to the text is where silent labelling bugs live - a total of 25.00 that also
    appears as a line amount would tag both.
    """
    words, labels = [], []

    def emit(text, label=None):
        parts = str(text).split()
        for i, part in enumerate(parts):
            words.append(part)
            if label is None:
                labels.append("O")
            else:
                labels.append(("B-" if i == 0 else "I-") + label)

    code_, symbol = rng.choice(CURRENCIES)
    date_style = rng.randrange(4)
    issued = date(2026, 1, 1) + timedelta(days=rng.randrange(360))
    due = issued + timedelta(days=rng.choice([14, 30, 45, 60]))

    emit(rng.choice(VENDORS), "VENDOR_NAME")
    emit(f"{rng.randrange(1, 200)} {rng.choice(['Fleet Street', 'Mill Road', 'Kings Way'])}")
    emit("INVOICE")

    emit(rng.choice(NUMBER_LABELS))
    emit(f"INV-{issued.year}-{rng.randrange(1000, 9999)}", "INVOICE_NUMBER")

    emit(rng.choice(DATE_LABELS))
    emit(a_date(issued, date_style), "ISSUE_DATE")

    if rng.random() < 0.85:
        emit(rng.choice(DUE_LABELS))
        emit(a_date(due, date_style), "DUE_DATE")

    emit("Bill To:")
    emit(rng.choice(BUYERS), "BUYER_NAME")

    emit("Description Qty Unit Price Amount")

    subtotal = 0.0
    for _ in range(rng.randrange(1, 7)):
        qty = rng.randrange(1, 30)
        unit = round(rng.uniform(4, 900), 2)
        amount = round(qty * unit, 2)
        subtotal += amount
        emit(rng.choice(GOODS), "LINE_DESCRIPTION")
        emit(str(qty), "LINE_QUANTITY")
        emit(money(unit, symbol), "LINE_UNIT_PRICE")
        emit(money(amount, symbol), "LINE_AMOUNT")

    subtotal = round(subtotal, 2)
    tax = round(subtotal * rng.choice([0.0, 0.05, 0.2]), 2)
    total = round(subtotal + tax, 2)

    emit("Subtotal")
    emit(money(subtotal, symbol), "SUBTOTAL")
    emit(rng.choice(["VAT", "Tax", "Sales Tax"]))
    emit(money(tax, symbol), "TAX")
    emit(rng.choice(TOTAL_LABELS))
    emit(money(total, symbol), "TOTAL")
    emit("Currency")
    emit(code_, "CURRENCY")

    return words, labels


rng = random.Random(20260901)   # fixed, so a rerun trains on the same data
examples = [generate_invoice(rng) for _ in range(4000)]
print(f"{len(examples)} invoices")
print("first 30 tokens of one:")
w, l = examples[0]
for word, label in list(zip(w, l))[:30]:
    print(f"  {word:22} {label}")


**Optional but strongly recommended.** Upload real or public sample invoices with labels
and mix them in here. Generated data teaches the generator; a handful of real documents is
what stops the numbers being about this notebook rather than about invoices.

Public sets worth looking at: SROIE, CORD, FUNSD. Leave this cell alone to train on
generated data only, and say so honestly wherever the accuracy figure ends up.

In [ ]:
# Mix in real examples if you have them, as a list of (words, labels) in the same shape.
real_examples = []
# from google.colab import files; files.upload()

all_examples = examples + real_examples
print(f"{len(all_examples)} total, {len(real_examples)} of them real")


## 2. Tokenize and align

A word can become several word-pieces. The label goes on the first piece, and the rest get
-100 so the loss ignores them. Getting this wrong is the classic silent bug in token
classification: it trains, the loss falls, and the model is learning the wrong thing.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def encode(batch):
    encoded = tokenizer(
        batch["words"], is_split_into_words=True,
        truncation=True, max_length=512, padding=False,
    )
    all_labels = []
    for i, labels in enumerate(batch["labels"]):
        word_ids = encoded.word_ids(batch_index=i)
        aligned, previous = [], None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)                 # special token
            elif word_id != previous:
                aligned.append(LABEL_TO_ID[labels[word_id]])
            else:
                aligned.append(-100)                 # continuation word-piece
            previous = word_id
        all_labels.append(aligned)
    encoded["labels"] = all_labels
    return encoded


dataset = Dataset.from_dict({
    "words": [w for w, _ in all_examples],
    "labels": [l for _, l in all_examples],
})
split = dataset.train_test_split(test_size=0.15, seed=20260901)
tokenized = split.map(encode, batched=True, remove_columns=["words", "labels"])
print(tokenized)


## 3. Train

DistilBERT, six epochs. Small enough to fine-tune on a free T4 in a few minutes and small
enough to serve on a CPU afterwards, which matters because the pipeline runs in a container
with no GPU.

In [ ]:
import numpy as np
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)
from seqeval.metrics import classification_report, f1_score

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id=LABEL_TO_ID,
)


def compute_metrics(evaluation):
    logits, labels = evaluation
    predictions = np.argmax(logits, axis=-1)
    true, predicted = [], []
    for prediction_row, label_row in zip(predictions, labels):
        true.append([LABELS[l] for l in label_row if l != -100])
        predicted.append([LABELS[p] for p, l in zip(prediction_row, label_row) if l != -100])
    return {"f1": f1_score(true, predicted)}


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="out",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=6,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    ),
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()


## 4. Per-field results

The overall F1 is the least useful number here. What matters is the per-field breakdown -
which fields it gets right and which it does not - because that is what tells you where the
pipeline will actually send documents to review, and it is what the README should carry.

Expect `LINE_*` fields to be the weak ones. They are the hardest and they are also where
most of the value is.

In [ ]:
import numpy as np
from seqeval.metrics import classification_report

output = trainer.predict(tokenized["test"])
predictions = np.argmax(output.predictions, axis=-1)

true, predicted = [], []
for prediction_row, label_row in zip(predictions, output.label_ids):
    true.append([LABELS[l] for l in label_row if l != -100])
    predicted.append([LABELS[p] for p, l in zip(prediction_row, label_row) if l != -100])

print(classification_report(true, predicted, digits=3))
print()
print("Copy this table into NOTES.md with the date, the training set size, and whether any")
print("real documents were in it. A number with no provenance is not evidence.")


## 5. Try it on one document

The same shape the pipeline will see: raw text in, tagged spans out.

In [ ]:
from transformers import pipeline as hf_pipeline

tagger = hf_pipeline("token-classification", model=model, tokenizer=tokenizer,
                     aggregation_strategy="simple", device=0 if model.device.type == "cuda" else -1)

sample = """Northgate Supplies
82 Mill Road
INVOICE
Invoice No. INV-2026-7741
Date of Issue: 04 March 2026
Due: 18/03/2026
Bill To: Orchard Foods
Description Qty Unit Price Amount
Consulting hours 12 GBP 85.00 GBP 1,020.00
Subtotal GBP 1,020.00
VAT GBP 204.00
Grand Total GBP 1,224.00
Currency GBP"""

for span in tagger(sample):
    print(f"{span['entity_group']:18} {span['score']:.3f}  {span['word']}")


## 6. Export

Saves the weights, zips them, and downloads. On the machine running mailman, unzip into
`models/extractor/` and set `MAILMAN_EXTRACTOR=trained`.

The zip is around 250 MB. It does **not** belong in git, and it will not fit a free hosting
tier's memory - `mailman/.gitignore` already excludes `models/`. The heuristic extractor
stays the deployed one; this is the local and showcase path.

In [ ]:
import shutil

model.save_pretrained("models/extractor")
tokenizer.save_pretrained("models/extractor")
shutil.make_archive("mailman-extractor", "zip", "models/extractor")

import os
size_mb = os.path.getsize("mailman-extractor.zip") / 1_000_000
print(f"mailman-extractor.zip  {size_mb:.0f} MB")

try:
    from google.colab import files
    files.download("mailman-extractor.zip")
except ImportError:
    print("Not on Colab - the zip is in the working directory.")


## 7. On the machine running mailman

```powershell
# unzip the download into the model directory
Expand-Archive mailman-extractor.zip -DestinationPath .\models\extractor -Force

# switch the pipeline over
$env:MAILMAN_EXTRACTOR = "trained"
docker compose up -d --build

# and check which extractor answered
curl.exe -F "file=@corpus\some-invoice.pdf" http://localhost:8000/documents
Invoke-RestMethod http://localhost:8000/documents/<id>/extraction | ConvertTo-Json -Depth 10
```

`model_name` on the extraction row says `trained:extractor` rather than `heuristic`. Because
extractions are append-only, the same document can be run through both and the two answers
compared directly - which is what the evaluation harness in stage 8 does across the whole
corpus.

## What to write down

The per-field table from section 4, with the date, how many training examples, and whether
any of them were real. Then the same corpus through the heuristic extractor, and the gap
between the two. That gap is the result worth reporting - not the F1 on its own, which only
says the model learned the generator.

## Known limits of this model

- Trained on generated invoices, so it has seen one author's idea of a layout.
- Text only. It never sees the page, so a two-column layout that interleaves in the text
  layer is invisible to it. LayoutLMv3 with bounding boxes from pdfplumber is the upgrade,
  and it is a real one.
- 512 word-pieces. A long multi-page invoice is truncated, silently.
- It tags spans; it does not understand. It cannot infer a total that was never printed,
  and it should not - the validation rules check arithmetic in Python for exactly that
  reason.
